In [239]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

torch.manual_seed(42)

In [240]:
# Download the data
!gdown --id 1VsikqoGtx6Ei12NIcmaIS4AYTwiksPJI

c:\Users\Rishabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\gdown\__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1VsikqoGtx6Ei12NIcmaIS4AYTwiksPJI
To: c:\Users\Rishabh\OneDrive\Desktop\IIT_Madras\Notes\IITM-BS-Programming-and-Data-Science\Degree\Deep Learning\preprocessed_yelp_data.csv

  0%|          | 0.00/94.8k [00:00<?, ?B/s]
100%|██████████| 94.8k/94.8k [00:00<00:00, 595kB/s]
100%|██████████| 94.8k/94.8k [00:00<00:00, 595kB/s]


In [241]:
# Read the data
data = pd.read_csv('preprocessed_yelp_data.csv')

In [242]:
# Split the data, 20% for testing
train, test = train_test_split(data, test_size=0.2, random_state=42)
train.shape, test.shape

((800, 2), (200, 2))

In [243]:
def build_vocab(texts, max_size=10000, min_freq=2):
    word_freq = {}
    vocab = {'<unknown>': 0, '<pad>': 1}
    for text in texts:
        for word in text:
            if word not in word_freq:
                word_freq[word] = 1
            else: word_freq[word] += 1
    
    size = 2
    for word, freq in word_freq.items():
        if freq >= min_freq:
            vocab[word] = size
            size += 1
    
    return vocab

In [244]:
import ast
train_texts = train['text'].tolist()
train_texts = [ast.literal_eval(text) for text in train_texts]
test_texts = test['text'].tolist()
test_texts = [ast.literal_eval(text) for text in test_texts]
vocab = build_vocab(train_texts)
len(vocab)

762

In [245]:
def numericalize_text(texts, vocab):
    return [[vocab.get(token, 0) for token in text] for text in texts]

In [246]:
train_num_texts = numericalize_text(train_texts, vocab)
test_num_texts = numericalize_text(test_texts, vocab)
train_num_texts[0]

[2, 3, 4, 2, 5, 6]

In [247]:
cur_max = -1
for text in train_num_texts:
    cur_max = max(cur_max, max(text))
cur_max

761

In [248]:
class YelpDataset(Dataset):
    def __init__(self, dataframe, max_seq_len):
        self.data = dataframe
        self.max_seq_len = max_seq_len
    
    def fix_size(self):
        final = []
        for i in self.data.index:
            if len(self.data['text'][i]) > self.max_seq_len:
                self.data['text'][i] = self.data['text'][i][:self.max_seq_len]
            elif len(self.data['text'][i]) < self.max_seq_len:
                self.data['text'][i] += [1] * (self.max_seq_len - len(self.data['text'][i]))
            temp = [torch.tensor(self.data['text'][i]), torch.tensor(self.data['label'][i])]
            final.append(temp)
        
        return final

In [249]:
train['text'] = train_num_texts
test['text'] = test_num_texts
trainYELP = YelpDataset(train, 100)
testYELP = YelpDataset(test, 100)
train_data = trainYELP.fix_size()
test_data = testYELP.fix_size()
cur_max = -1
for i in range(len(train_data)):
    cur_max = max(cur_max, train_data[i][0].max())
train_data[0], cur_max

C:\Users\Rishabh\AppData\Local\Temp\ipykernel_5336\1249899786.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.data['text'][i] += [1] * (self.max_seq_len - len(self.data['text'][i]))
C:\Users\Rishabh\AppData\Local\Temp\ipykernel_533

([tensor([2, 3, 4, 2, 5, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1]),
  tensor(0)],
 tensor(761))

In [250]:
trainLoader = DataLoader(train_data, batch_size=32, shuffle=True)
testLoader = DataLoader(test_data, batch_size=32, shuffle=False)

In [251]:
class RNNModule(nn.Module):
    def __init__(self, input_size,  embedding_dim, hidden_size, num_layers, output_size):
        super(RNNModule, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.h0 = torch.zeros(self.num_layers, embedding_dim, self.hidden_size)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x, self.h0)
        out = self.linear(out[:, -1, :])
        return out

In [252]:
model = RNNModule(len(vocab), 100, 256, 2, 1)
# Number of parameters
sum(p.numel() for p in model.parameters())

299689

In [253]:
critereon = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [254]:
def train_model():
    model.train()
    for epoch in range(2):
        for i, (texts, labels) in enumerate(trainLoader):
            #print(texts, labels)
            
            optimizer.zero_grad()
            outputs = model(texts)
            loss = critereon(outputs, labels.unsqueeze(1).float())
            loss.backward()
            optimizer.step()
            if i % 100 == 0:
                print(f'Epoch {epoch}, Iteration {i}, loss: {loss.item()}')
        print(f'Epoch {epoch}, loss: {loss.item()}')

def evaluate_model():
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for texts, labels in testLoader:
            outputs = model(texts)
            predictions = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predictions == labels.unsqueeze(1)).sum().item()
        print(f'Accuracy: {correct/total}')

In [255]:
train_model()

Epoch 0, Iteration 0, loss: 0.7066386938095093
Epoch 0, loss: 0.6669815182685852
Epoch 1, Iteration 0, loss: 0.7209752202033997
Epoch 1, loss: 0.6923879384994507


In [256]:
evaluate_model()

Accuracy: 0.525
